# Assignment 3 — Does Quantization Make Code LLMs Brittle?

**NC State · CSC 591/791 · Trustworthy AI for Software Engineering · individual work**

Quantization papers report one headline number — usually pass@1 on clean, canonical
prompts — and call a 4-bit model "near-lossless" when that number barely moves. Real
prompts are not canonical: a function gets renamed, a docstring gets reworded, the
indentation changes. **When the clean score holds, does robustness hold too?**

In this assignment you will

1. **quantize** one code LLM yourself, four ways;
2. **evaluate** every version on the same robustness benchmarks, before and after quantization;
3. **analyse** whether quantization costs robustness that the clean score does not show.

There is **no answer key**. Nobody knows in advance whether 4-bit quantization hurts the
robustness of this model, so you are graded on whether your protocol is right, your
results are reproducible, and your claims match your evidence — not on reaching a
particular conclusion. A well-supported *"no significant change"* earns full credit.

### What is provided and what you write

The notebook provides the plumbing — data download, prompt construction, batched
generation with caching, sandboxed test execution, and the results/sample loggers. Cells
marked **`TODO`** are yours. Each TODO says what its function must return so that the
provided cells after it keep working.

| Part | What you do | Points |
|---|---|---|
| 1 | Load `fp16`, `int8`, `nf4`; run GPTQ yourself to produce `gptq4` | 20 |
| 2 | **E0** clean baseline, all four configurations | 10 |
| 3 | **E1** ReCode robustness, all four perturbation families | 25 |
| 4 | **E2** *or* **E3** (both → bonus) | 15 |
| 5 | Paired statistics and uncertainty | 10 |
| 6 | Report: the confound handled, flipped examples explained, claims matched to evidence | 15 |
| — | Submission: notebook with all outputs, run top to bottom; `README.md` explains your code | 5 |
| bonus | Both E2 and E3, or a fifth configuration (e.g. AWQ) | +5 |

### Rules

- Use a **GPU runtime**: *Runtime → Change runtime type → T4 GPU*.
- **Do not change** the model, the decoding settings, the prompt template (except inside E3),
  or the ReCode subset. They are fixed so that everyone's numbers are comparable.
- **Never use HumanEval or any ReCode file as GPTQ calibration data.** That leaks the test set
  into the quantized weights.
- AI assistance is allowed. Say in the report what you used it for; you remain responsible
  for every number.
- Submit two files to Gradescope: this notebook **with its outputs** (`.ipynb`) and a `README.md`
  explaining your code. Part 7 says what goes in each.

## Reading

| For | Paper |
|---|---|
| E1 | Wang et al., *ReCode: Robustness Evaluation of Code Generation Models*, ACL 2023 — [arXiv:2212.10264](https://arxiv.org/abs/2212.10264). Read §3 for the perturbation families and §4 for the metrics. |
| E1 (quantization) | Wei et al., *Greener yet Powerful: Taming Large Code Generation Models with Quantization* — [arXiv:2303.05378](https://arxiv.org/abs/2303.05378). They ran ReCode on quantized code models; compare your findings with theirs. |
| E2 | Fang et al., *EVALOOOP: A Self-Consistency-Centered Framework for Assessing Large Language Model Robustness in Programming* — [arXiv:2505.12185](https://arxiv.org/abs/2505.12185). |
| E3 | Sclar et al., *Quantifying Language Models' Sensitivity to Spurious Features in Prompt Design*, ICLR 2024 — [arXiv:2310.11324](https://arxiv.org/abs/2310.11324). |
| GPTQ | Frantar et al., *GPTQ*, ICLR 2023 — [arXiv:2210.17323](https://arxiv.org/abs/2210.17323). |
| int8 / NF4 | Dettmers et al., *LLM.int8()* — [arXiv:2208.07339](https://arxiv.org/abs/2208.07339); *QLoRA* (for NF4) — [arXiv:2305.14314](https://arxiv.org/abs/2305.14314). |

## 0. Setup

Run the install cell once. **If it changes the installed `torch`, restart the session
(*Runtime → Restart session*) and continue from the cell after it.** The next cell checks.

In [ ]:
!pip install -q "llmcompressor==0.14.0" "transformers==5.17.0" "bitsandbytes>=0.46" scipy

In [ ]:
import importlib.metadata as md
for pkg in ["torch", "transformers", "llmcompressor", "compressed-tensors", "bitsandbytes"]:
    print(f"{pkg:20s} {md.version(pkg)}")

import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print(torch.cuda.get_device_name(0))

### Configuration

`QUICK = True` runs every evaluation on the first `N_QUICK` problems so you can debug the
whole notebook in minutes. **Set `QUICK = False` for the run you submit.**

Everything is written to the Colab VM's local disk (`/content/csc591-a3`). Generations are
cached there, so re-running a cell after an error resumes where it stopped. **The local disk
is wiped when the runtime is disconnected or deleted**, so keep the browser tab open during the
full run, and download the notebook as soon as it finishes.

In [ ]:
import os

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # fixed for the whole class
DEVICE = "cuda"
DTYPE = "float16"            # T4 has no fast bfloat16
QUICK = True                 # <- set to False for the graded run
N_QUICK = 20
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16              # lower it if you run out of GPU memory
EXEC_TIMEOUT = 10            # seconds per test program

OUT_DIR = "/content/csc591-a3" if os.path.isdir("/content") else "./a3_out"

DATA_DIR = os.path.join(OUT_DIR, "data")
GEN_DIR = os.path.join(OUT_DIR, "generations")
FIG_DIR = os.path.join(OUT_DIR, "figures")
GPTQ_DIR = os.path.join(OUT_DIR, "gptq4-checkpoint")
for d in (DATA_DIR, GEN_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
print("outputs ->", OUT_DIR)

In [ ]:
import gc, json, sys, time, ast, re, random, hashlib, shutil, subprocess, tempfile, urllib.request
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig

TORCH_DTYPE = getattr(torch, DTYPE)
random.seed(0); np.random.seed(0); torch.manual_seed(0)

## 1. Data: HumanEval and the ReCode perturbations

ReCode publishes its perturbed HumanEval files, so we download them rather than
regenerate them — everyone scores exactly the same prompts.

Two things about these files matter for your analysis:

- The **docstring** and **function-name** families perturb the ordinary HumanEval prompt.
  Their nominal (unperturbed) reference is `HumanEval.jsonl`.
- The **syntax** and **format** families perturb *partial code*: the prompt already
  contains the first half of the reference solution, and the model completes the rest.
  Their nominal reference is `humaneval_partial.jsonl`, **not** the ordinary prompt. Compare
  each perturbation with its own nominal set.

ReCode ships many transformations, several with up to 10 random seeds. To keep the compute
within a free T4, the class uses the fixed subset below: two transformations per family, seed 0.

In [ ]:
RECODE_BASE = "https://raw.githubusercontent.com/amazon-science/recode/main/dataset-release"

NOMINAL_FILES = {
    "nominal":         "nominal/HumanEval.jsonl",
    "nominal_partial": "nominal/humaneval_partial.jsonl",
}

# family -> {variant name: file}. Fixed for the whole class; do not change.
RECODE_SUBSET = {
    "docstring": {
        "BackTranslation":     "nlaugmenter/humaneval_BackTranslation_s0.jsonl",
        "ButterFingers":       "nlaugmenter/humaneval_ButterFingersPerturbation_s0.jsonl",
    },
    "func_name": {
        "CamelCase":           "func_name/humaneval_FuncRenameCamelCase_s0.jsonl",
        "ButterFinger":        "func_name/humaneval_FuncRenameButterFinger_s0.jsonl",
    },
    "syntax": {
        "VarRenamerCB":        "natgen/humaneval_VarRenamerCB_s0.jsonl",
        "DeadCodeInserter":    "natgen/humaneval_DeadCodeInserter_s0.jsonl",
    },
    "format": {
        "TabIndent":           "format/humaneval_tab_indent_s0.jsonl",
        "Doc2Comments":        "format/humaneval_doc2comments_s0.jsonl",
    },
}
# which nominal set each family is compared against
FAMILY_NOMINAL = {"docstring": "nominal", "func_name": "nominal",
                  "syntax": "nominal_partial", "format": "nominal_partial"}


def fetch(rel_path):
    local = os.path.join(DATA_DIR, rel_path.replace("/", "__"))
    if not os.path.exists(local):
        if rel_path.startswith("nominal/"):
            url = f"{RECODE_BASE}/{rel_path}"
        else:
            url = f"{RECODE_BASE}/perturbed_finalized/humaneval/full/{rel_path}"
        urllib.request.urlretrieve(url, local)
    with open(local) as f:
        rows = [json.loads(line) for line in f]
    assert len(rows) == 164, (rel_path, len(rows))
    return rows


DATASETS = {name: fetch(path) for name, path in NOMINAL_FILES.items()}
for family, variants in RECODE_SUBSET.items():
    for variant, path in variants.items():
        DATASETS[f"{family}/{variant}"] = fetch(path)

TASK_IDS = [r["task_id"] for r in DATASETS["nominal"]]
if QUICK:
    TASK_IDS = TASK_IDS[:N_QUICK]
print(len(DATASETS), "datasets,", len(TASK_IDS), "problems per dataset")

Look at one perturbed problem next to its nominal version before you go on. Pick a few
`task_id`s and families and read the prompts. You will need this to explain flipped examples
in your report.

In [ ]:
def show(dataset, task_id="HumanEval/0"):
    row = next(r for r in DATASETS[dataset] if r["task_id"] == task_id)
    print(f"--- {dataset}  entry_point={row['entry_point']}")
    print(row["prompt"])

show("nominal_partial")
show("syntax/VarRenamerCB")

## 2. Prompting, generation, and test execution (provided)

**Prompt template (fixed).** The model sees the code in a user message, and its answer is
*prefilled* with that same code, so it continues the function exactly where the prompt
stops. This makes the instruct model behave like a completion model, which is what ReCode's
partial-code perturbations assume. Generation stops at the closing code fence.

**Decoding (fixed).** Greedy, `max_new_tokens=512`, no repetition penalty. `generate()`
fills any setting we leave unset from the model's own `generation_config`, which for Qwen
includes sampling settings and `repetition_penalty=1.1`, so every one of them is set
explicitly. The only thing that differs between your runs is the weights.

**Execution.** `prompt + completion` is first *sanitized*: only imports, function and class
definitions, and plain assignments are kept, so a self-test the model appended (often with
wrong expected values) cannot fail the program before the real tests run. The HumanEval tests
and `check(entry_point)` are then appended, and the program runs in a separate Python process
with a timeout. `SAMPLES` keeps the raw completion. Generated code is untrusted; it runs inside your
Colab VM, which is disposable.

In [ ]:
INSTRUCTION = "Complete the following Python function. Keep the function name and signature unchanged."
FENCE = "`" * 3
CHAT_STOP = [FENCE]


def chat_input(tok, prompt, instruction=INSTRUCTION):
    """Chat-formatted input whose assistant turn is prefilled with the code prompt."""
    body = prompt if prompt.endswith("\n") else prompt + "\n"
    user = f"{instruction}\n\n{FENCE}python\n{body}{FENCE}"
    text = tok.apply_chat_template([{"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)
    return text + f"{FENCE}python\n" + prompt


def truncate_at_stop(text, stop):
    cut = min([i for s in stop for i in [text.find(s)] if i >= 0], default=len(text))
    return text[:cut]


GEN_STATS = defaultdict(lambda: {"tokens": 0, "seconds": 0.0})


@torch.inference_mode()
def generate(model, tok, inputs, stop, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE, stats_key=None):
    """Greedy batched generation. Returns one completion string per input, cut at the first stop string."""
    # generate() fills every field left unset here from the model's own generation_config, which
    # for Qwen includes repetition_penalty=1.1 and sampling settings. Set them all explicitly.
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens, do_sample=False, stop_strings=stop or None,
        repetition_penalty=1.0, temperature=1.0, top_p=1.0, top_k=50,
        pad_token_id=tok.pad_token_id, eos_token_id=model.generation_config.eos_token_id)
    order = sorted(range(len(inputs)), key=lambda i: len(inputs[i]), reverse=True)
    outputs = [None] * len(inputs)
    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        enc = tok([inputs[i] for i in idx], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        t0 = time.time()
        out = model.generate(**enc, generation_config=gen_config, tokenizer=tok)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        new = out[:, enc["input_ids"].shape[1]:]
        if stats_key:
            GEN_STATS[stats_key]["seconds"] += time.time() - t0
            GEN_STATS[stats_key]["tokens"] += int((new != tok.pad_token_id).sum())
        for i, text in zip(idx, tok.batch_decode(new, skip_special_tokens=True)):
            outputs[i] = truncate_at_stop(text, stop) if stop else text
    return outputs

In [ ]:
class GenCache:
    """Append-only cache of generations, one JSONL file per configuration.

    Entries are keyed by a hash of the exact model input and generation settings, so a changed
    prompt is regenerated and an identical one (e.g. E0 and E2 loop 1) is generated only once.
    """

    def __init__(self, config_id):
        self.path = os.path.join(GEN_DIR, f"{config_id}.jsonl")
        self.data = {}
        if os.path.exists(self.path):
            with open(self.path) as f:
                for line in f:
                    rec = json.loads(line)
                    self.data[rec["key"]] = rec["output"]

    DECODING = "greedy-rp1.0"   # part of the key: change it if you change the decoding settings

    @classmethod
    def key(cls, text, stop, max_new_tokens):
        return hashlib.sha1(json.dumps([text, stop, max_new_tokens, cls.DECODING]).encode()).hexdigest()

    def run(self, model, tok, inputs, stop, stats_key=None, chunk=64, max_new_tokens=MAX_NEW_TOKENS):
        keys = [self.key(x, stop, max_new_tokens) for x in inputs]
        todo = [(k, x) for k, x in dict(zip(keys, inputs)).items() if k not in self.data]
        for start in range(0, len(todo), chunk):
            part = todo[start:start + chunk]
            outs = generate(model, tok, [x for _, x in part], stop, stats_key=stats_key,
                            max_new_tokens=max_new_tokens)
            with open(self.path, "a") as f:
                for (k, _), o in zip(part, outs):
                    self.data[k] = o
                    f.write(json.dumps({"key": k, "output": o}) + "\n")
            print(f"  generated {min(start + chunk, len(todo))}/{len(todo)}", end="\r")
        return [self.data[k] for k in keys]

In [ ]:
def sanitize(code):
    """Keep imports, definitions and call-free assignments; drop top-level code the model added
    (its own asserts, prints, `if __name__ == "__main__":` blocks). Unparsable code is returned
    unchanged and fails at execution."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code
    defs = (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
    tree.body = [n for n in tree.body if isinstance(n, defs) or
                 (isinstance(n, (ast.Assign, ast.AnnAssign))
                  and not any(isinstance(c, ast.Call) for c in ast.walk(n)))]
    return ast.unparse(tree)


def build_program(prompt, completion, row):
    return f"{sanitize(prompt + completion)}\n\n{row['test']}\n\ncheck({row['entry_point']})\n"


def run_program(program, timeout=EXEC_TIMEOUT):
    """Run a program in a fresh interpreter. Returns (passed, status)."""
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "prog.py")
        with open(path, "w") as f:
            f.write(program)
        try:
            r = subprocess.run([sys.executable, path], cwd=tmp, capture_output=True,
                               text=True, timeout=timeout)
        except subprocess.TimeoutExpired:
            return False, "timeout"
    if r.returncode == 0:
        return True, "passed"
    err = r.stderr.strip().splitlines()
    return False, (err[-1][:200] if err else f"exit {r.returncode}")


def run_programs(programs, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return list(ex.map(run_program, programs))


# sanity check: the reference solutions must pass
_rows = [r for r in DATASETS["nominal"] if r["task_id"] in TASK_IDS]
_res = run_programs([build_program(r["prompt"], r["canonical_solution"], r) for r in _rows])
print("reference solutions passing:", sum(p for p, _ in _res), "/", len(_rows))

In [ ]:
SAMPLES = []      # one dict per generation (prompt, raw output, verdict)
RESULTS = []      # one dict per metric
CONFIG_META = {
    "fp16":  {"method": "none",               "bits": 16},
    "int8":  {"method": "bnb-llm.int8",       "bits": 8},
    "nf4":   {"method": "bnb-nf4",            "bits": 4},
    "gptq4": {"method": "gptq-w4a16-g128",    "bits": 4},
}


def record_result(config, eval_id, variant, metric, n, value, ci_low=None, ci_high=None):
    meta = CONFIG_META[config]
    RESULTS.append({"config": config, "method": meta["method"], "bits": meta["bits"],
                    "eval": eval_id, "variant": variant, "metric": metric, "n": n,
                    "value": value, "ci_low": ci_low, "ci_high": ci_high})


def evaluate_dataset(config, model, tok, cache, dataset, eval_id, instruction=INSTRUCTION, variant=None):
    """Generate and test every problem of one dataset. Returns {task_id: passed}."""
    rows = [r for r in DATASETS[dataset] if r["task_id"] in TASK_IDS]
    variant = variant or dataset
    inputs = [chat_input(tok, r["prompt"], instruction) for r in rows]
    outs = cache.run(model, tok, inputs, CHAT_STOP, stats_key=config)
    verdicts = run_programs([build_program(r["prompt"], o, r) for r, o in zip(rows, outs)])
    passed = {}
    for r, o, (ok, status) in zip(rows, outs, verdicts):
        passed[r["task_id"]] = ok
        SAMPLES.append({"config": config, "eval": eval_id, "variant": variant, "item_id": r["task_id"],
                        "prompt": r["prompt"], "output": o, "passed": ok, "status": status})
    return passed

## 3. Part 1 — Quantize the model (20 pts)

| ID | Precision | Method | Calibration data |
|---|---|---|---|
| `fp16` | 16-bit | none — the baseline | — |
| `int8` | 8-bit | LLM.int8() via `bitsandbytes` | none |
| `nf4` | 4-bit | NF4 via `bitsandbytes` | none |
| `gptq4` | 4-bit weights, 16-bit activations | GPTQ, group size 128, via `llmcompressor` | **you choose** |

`int8` and `nf4` quantize at load time. `gptq4` is the one where you actually run a
quantization algorithm: choose a calibration set, run GPTQ, save the checkpoint, and reload it.

### TODO 1a — `load_model`

In [ ]:
def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    tok.padding_side = "left"          # required for batched generation with a decoder-only model
    return tok


def load_model(config):
    """Return the model for `config` in eval mode on the GPU.

    fp16  : the original weights in float16
    int8  : LLM.int8() through BitsAndBytesConfig
    nf4   : 4-bit NF4 through BitsAndBytesConfig, computing in float16
    gptq4 : the checkpoint you saved to GPTQ_DIR in TODO 1b
    """
    # TODO: implement the four cases. Hints: AutoModelForCausalLM.from_pretrained(...,
    # dtype=TORCH_DTYPE, device_map=DEVICE, quantization_config=...). Look up the
    # BitsAndBytesConfig fields for 8-bit and for 4-bit NF4.
    raise NotImplementedError("TODO 1a")

### TODO 1b — run GPTQ

Use `llmcompressor`'s `oneshot` with a `GPTQModifier` whose scheme is `"W4A16"` (4-bit
symmetric integer weights, group size 128, 16-bit activations). Leave `lm_head` unquantized.
Save with `save_compressed=True`, and save the tokenizer next to it.

**Calibration data is your design decision; justify it in the report.** GPTQ chooses each
layer's rounding to minimise the error on the calibration activations, so what you
calibrate on may matter. Some options: code instructions (e.g.
`theblackcat102/evol-codealpaca-v1`), general text (e.g. `wikitext`), or chat data. Record
the dataset, number of samples and sequence length. **HumanEval and ReCode are forbidden.**

**Load the model in float32 for calibration**, not float16. In float16 the T4 run overflows
inside the network, the Hessians become invalid, and llmcompressor quietly quantizes almost
every layer with plain round-to-nearest instead of GPTQ — you would be comparing NF4 against
RTN without knowing it. The cell below checks for this and refuses to keep such a checkpoint.
(Evaluation still loads the saved checkpoint in float16.)

On a T4, 256 samples of 512 tokens takes a few minutes. The cell skips itself if a
checkpoint already exists.

In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from datasets import load_dataset

CALIB_DATASET = None        # TODO: name of the calibration dataset you chose
NUM_CALIB_SAMPLES = 256
CALIB_SEQ_LEN = 512


def build_calibration_set(tok):
    """Return a datasets.Dataset with tokenized `input_ids` and `attention_mask` columns."""
    raise NotImplementedError("TODO 1b: load, format and tokenize NUM_CALIB_SAMPLES samples")


def run_gptq():
    raise NotImplementedError("TODO 1b: load the model in float32, run oneshot with a GPTQModifier, save to GPTQ_DIR")


from loguru import logger as _llmc_logger


def run_gptq_checked(max_fallback=0.10):
    """Run GPTQ and fail if too many layers silently fell back to round-to-nearest."""
    warnings_seen = []
    sink = _llmc_logger.add(lambda m: warnings_seen.append(m.record["message"]), level="WARNING")
    try:
        run_gptq()
    finally:
        _llmc_logger.remove(sink)
    for msg in warnings_seen:
        m = re.search(r"Hessian inversion failed for (\d+)/(\d+) modules", msg)
        if m and int(m.group(1)) / int(m.group(2)) > max_fallback:
            shutil.rmtree(GPTQ_DIR, ignore_errors=True)
            raise RuntimeError(
                f"GPTQ fell back to round-to-nearest for {m.group(1)}/{m.group(2)} layers, so this "
                "checkpoint is not a GPTQ model. It was deleted. Check the calibration dtype and data.")
    print("GPTQ fallback check passed")


if not os.path.exists(os.path.join(GPTQ_DIR, "config.json")):
    t0 = time.time()
    run_gptq_checked()
    print(f"GPTQ done in {time.time() - t0:.0f}s")
else:
    print("using existing checkpoint at", GPTQ_DIR)

### Efficiency (provided)

Record what each configuration buys you. Note one subtlety for your report: `transformers`
stores the GPTQ checkpoint compressed on disk, but on a T4 it **decompresses the weights to
fp16 on the first forward pass**. Its numbers are the numbers of 4-bit weights, but its GPU
memory and speed are not what a dedicated int4 kernel (e.g. vLLM) would give.

In [ ]:
def dir_size_gb(path):
    return sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(path) for f in fs) / 1e9


def efficiency_row(config, model):
    row = {"config": config, "weights_in_memory_gb": model.get_memory_footprint() / 1e9}
    if config == "gptq4":
        row["checkpoint_on_disk_gb"] = dir_size_gb(GPTQ_DIR)
    if torch.cuda.is_available():
        row["peak_gpu_memory_gb"] = torch.cuda.max_memory_allocated() / 1e9
    s = GEN_STATS[config]
    row["tokens_per_second"] = s["tokens"] / s["seconds"] if s["seconds"] else None
    return row


EFFICIENCY = []

### Sanity check before the long runs

Load one configuration and look at its output on two problems. If the completion is empty
or garbled, fix that before you spend GPU time.

In [ ]:
tok = load_tokenizer()
_model = load_model("fp16")
_rows = [r for r in DATASETS["nominal"]][:2]
for r, o in zip(_rows, generate(_model, tok, [chat_input(tok, r["prompt"]) for r in _rows], CHAT_STOP)):
    print("=" * 60, r["task_id"]); print(r["prompt"] + o)
del _model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Parts 2–4 — Run the evaluations

The loop below loads one configuration at a time, runs **E0** and **E1** on it, and runs
**E2** and/or **E3** if you set the flags. Everything is cached, so if the session dies,
re-run the setup cells and this cell; finished generations are not regenerated.

**Time budget (measured on a free Colab T4, `QUICK = False`):** the GPTQ step takes about
10 minutes, and E0 + E1 + E2 over all four configurations about **2.5 hours**. E3 adds
roughly another hour (estimate). `int8` (LLM.int8()) generates about 3× slower than the
others and dominates the total. Do the `QUICK` run first, then start the full run early —
on a free runtime that may mean a separate session.

The `TODO` functions this loop calls are defined in the cells after it — run those
cells first, then come back to this one. E0 and E1 need nothing from you to *run*; your
E1 work is the metric computation in Part 3.

In [ ]:
CONFIGS = ["fp16", "int8", "nf4", "gptq4"]
RUN_E2 = True     # EVALOOOP self-consistency loop  (do at least one of E2/E3)
RUN_E3 = False    # prompt-template sensitivity
E2_MAX_LOOPS = 3

PASS = defaultdict(dict)      # PASS[config][dataset or variant] = {task_id: passed}
E2_OUT = {}                   # E2_OUT[config] = output of run_evaloop

In [ ]:
def run_all():
    tok = load_tokenizer()
    for config in CONFIGS:
        print(f"===== {config}")
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        model = load_model(config)
        cache = GenCache(config)

        # E0: both nominal sets (the partial one is the reference for syntax/format)
        for ds in NOMINAL_FILES:
            PASS[config][ds] = evaluate_dataset(config, model, tok, cache, ds, "E0")
        # E1: ReCode subset
        for family, variants in RECODE_SUBSET.items():
            for variant in variants:
                name = f"{family}/{variant}"
                PASS[config][name] = evaluate_dataset(config, model, tok, cache, name, "E1")
        if RUN_E2:
            E2_OUT[config] = run_evaloop(config, model, tok, cache, E2_MAX_LOOPS)
        if RUN_E3:
            for tid, instruction in E3_TEMPLATES.items():
                PASS[config][f"E3/{tid}"] = run_template(config, model, tok, cache, tid, instruction)

        EFFICIENCY.append(efficiency_row(config, model))
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 5. Part 2 — E0 clean baseline (10 pts)

pass@1 with greedy decoding is just the fraction of problems whose single completion passes.
Report it for both nominal sets, with a 95% confidence interval (TODO 5 provides the
bootstrap you need). The helper below writes the rows.

In [ ]:
def pass_at_1(passed):
    return float(np.mean(list(passed.values())))


def record_e0():
    for config in CONFIGS:
        for ds in NOMINAL_FILES:
            p = PASS[config][ds]
            lo, hi = bootstrap_ci(list(p.values()))
            record_result(config, "E0", ds, "pass@1", len(p), pass_at_1(p), lo, hi)

## 6. Part 3 — E1 ReCode robustness metrics (25 pts)

### TODO 3 — `recode_metrics`

Implement the three ReCode metrics for one perturbation, comparing it with **its own nominal
set** (`FAMILY_NOMINAL`). With greedy decoding and one seed per transformation, ReCode §4
reduces to the following:

- **Robust Pass@1** — fraction of problems solved on the perturbed prompt.
- **Robust Drop@1** — `(nominal pass@1 − robust pass@1) / nominal pass@1`.
- **Robust Relative@1** — fraction of problems whose outcome *changes* in either direction
  (pass→fail or fail→pass).

Also return the **conditional robustness** this assignment requires: among problems the
configuration solves on the nominal prompt, the fraction it still solves under the
perturbation. Section 8 explains why you need it.

Then implement `family_worst_case`: a problem counts as robust for a family only if it
passes **every** transformation in that family. This is the per-family view.

In [ ]:
def recode_metrics(nominal, perturbed):
    """nominal, perturbed: {task_id: bool} over the same task_ids.

    Return a dict with keys: robust_pass, robust_drop, robust_relative, conditional_robust, n.
    """
    raise NotImplementedError("TODO 3")


def family_worst_case(config, family):
    """Return {task_id: bool}: True iff the problem passes every variant of `family`."""
    raise NotImplementedError("TODO 3")

In [ ]:
def record_e1():
    for config in CONFIGS:
        for family, variants in RECODE_SUBSET.items():
            nominal = PASS[config][FAMILY_NOMINAL[family]]
            for variant in list(variants) + ["worst_case"]:
                perturbed = (family_worst_case(config, family) if variant == "worst_case"
                             else PASS[config][f"{family}/{variant}"])
                m = recode_metrics(nominal, perturbed)
                for metric in ("robust_pass", "robust_drop", "robust_relative", "conditional_robust"):
                    record_result(config, "E1", f"{family}/{variant}", metric, m["n"], m[metric])

## 7. Part 4 — E2 *or* E3 (15 pts; both for the bonus)

### E2 — EVALOOOP self-consistency loop

EVALOOOP perturbs the model with its **own** output: code → natural-language description →
code → … until the code stops passing its tests. Robustness is the **Average Sustainable
Loops (ASL)**: the mean number of consecutive loops a problem survives. Read the paper for
the exact definition and reproduce it; document any difference.

Our adaptation, which you may refine but must document: the paper uses MBPP+ and ten
loops; we use the HumanEval problems already loaded, and `E2_MAX_LOOPS` = 3 loops.

Why only three: with greedy decoding the loop reaches a **fixed point** quickly. Once the
model writes the same code twice, it describes it the same way and regenerates the same code,
so a problem that survives loop 3 survives every later loop. In our pilot, loops 3, 4 and 5
gave identical pass rates for every configuration. Discuss in your report what this means
for ASL as a robustness measure under greedy decoding.

Helpers are provided. `strip_docstring` matters: if the model is shown the previous
docstring, it can copy it instead of describing the code, and the loop measures nothing.

In [ ]:
DESCRIBE_INSTRUCTION = (
    "Here is a Python function without its docstring. Write the docstring: say what the "
    "function does, what each parameter means, and what it returns, and give one or two "
    "examples in doctest format (>>> call, then the expected result).")
DOC_STOP = ['"""', FENCE]


def strip_docstring(code, entry_point):
    """Remove the docstring of `entry_point`. Returns None if the code does not parse."""
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return None
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == entry_point:
            body = node.body
            if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0], "value", None), ast.Constant) \
                    and isinstance(body[0].value.value, str):
                node.body = body[1:] or [ast.Pass()]
    return ast.unparse(tree)


def signature_header(original_prompt, entry_point):
    """The original prompt up to and including the signature of `entry_point` (imports and
    helper functions included), and the indentation of the function body."""
    lines = original_prompt.splitlines(keepends=True)
    start = next(i for i, l in enumerate(lines) if l.lstrip().startswith(f"def {entry_point}("))
    end = start
    while not lines[end].rstrip().endswith(":"):
        end += 1
    indent = " " * (len(lines[start]) - len(lines[start].lstrip()) + 4)
    return "".join(lines[:end + 1]), indent


def describe_input(tok, code, original_prompt, entry_point):
    """Ask for a docstring. The answer is prefilled up to the opening quotes, so the model can
    only write docstring text; generation stops at the closing quotes (DOC_STOP)."""
    header, indent = signature_header(original_prompt, entry_point)
    user = f"{DESCRIBE_INSTRUCTION}\n\n{FENCE}python\n{code}\n{FENCE}"
    text = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    return text + f'{FENCE}python\n{header}{indent}"""\n'


def prompt_from_description(original_prompt, entry_point, description):
    """The original header and signature, with the generated description as the docstring."""
    header, indent = signature_header(original_prompt, entry_point)
    lines = [l.strip() for l in description.strip("\n").splitlines()]
    doc = "\n".join(indent + l if l else "" for l in lines)
    return f'{header}{indent}"""\n{doc}\n{indent}"""\n'

### TODO 4a — `run_evaloop`

Loop 1 is the ordinary E0 generation from the original prompt. For each later loop, take the
previous loop's code (prompt + completion) for the problems that are still passing, strip its
docstring, ask the model to describe it (`describe_input`, stop strings `DOC_STOP`, 256 new
tokens), build a new prompt with `prompt_from_description`, generate code from it with
`chat_input`, and test it against the original tests. Use `cache.run` for every generation
so the work is cached. Append a `SAMPLES` row for
each generated program, with `eval="E2"` and `variant=f"loop{k}"`.

In [ ]:
def run_evaloop(config, model, tok, cache, max_loops):
    """Return {"survived": {task_id: int}, "pass_by_loop": [pass@1 at loop 1, ..., loop max_loops]}.

    survived[t] = number of consecutive loops, starting at loop 1, that problem t passes (0..max_loops).
    pass_by_loop[k-1] = fraction of all problems still passing at loop k.
    """
    raise NotImplementedError("TODO 4a")


def asl(survived):
    """Average Sustainable Loops."""
    raise NotImplementedError("TODO 4a")

### E3 — prompt-template sensitivity

Every template below asks for the same thing; only the wording and layout of the user
message change. (The assistant prefill is the same in all of them, so the model continues
the same code.) `T0` is the template E0 used, so its result is reused.

In [ ]:
E3_TEMPLATES = {
    "T0": INSTRUCTION,
    "T1": "Finish this function.",
    "T2": ("You are an expert Python programmer. Read the docstring carefully and implement the "
           "function so that it passes all hidden unit tests. Do not rename the function."),
    "T3": "### Task\nImplement the function below in Python.\n### Constraints\nKeep the signature.",
    "T4": "complete the python function:",
    "T5": "Please write the body of the function that follows, keeping its name and parameters as they are.",
}


def run_template(config, model, tok, cache, tid, instruction):
    if tid == "T0":
        return PASS[config]["nominal"]
    return evaluate_dataset(config, model, tok, cache, "nominal", "E3", instruction=instruction, variant=tid)

### TODO 4b — E3 spread

For each configuration, compute pass@1 under every template, and the **spread** (max − min
across templates), following Sclar et al.'s notion of format sensitivity. Also count, per
configuration, how many problems are *template-sensitive*: solved under some templates and
failed under others. Write the rows with `record_result(config, "E3", ...)`.

In [ ]:
def record_e3():
    raise NotImplementedError("TODO 4b")

In [ ]:
def record_e2():
    for config, out in E2_OUT.items():
        n = len(out["survived"])
        lo, hi = bootstrap_ci(list(out["survived"].values()))
        record_result(config, "E2", "all", "ASL", n, asl(out["survived"]), lo, hi)
        for k, p in enumerate(out["pass_by_loop"], 1):
            record_result(config, "E2", f"loop{k}", "pass@1", n, p)

## 8. Part 5 — Paired statistics (10 pts)

All configurations answer the **same** problems, so every comparison is paired. Comparing
two independent proportions throws that pairing away and overstates the noise.

### TODO 5 — `bootstrap_ci` and `paired_compare`

- `bootstrap_ci(values)`: a 95% percentile bootstrap interval for the mean of `values`
  (booleans or numbers), resampling problems with replacement, 10,000 resamples, fixed seed.
- `paired_compare(a, b)`: given two `{task_id: bool}` dicts over the same problems (e.g.
  `fp16` vs `nf4` on one perturbation), return the discordant counts (`a_only`: passed under
  A, failed under B; `b_only`: the reverse), the difference in pass rate, an **exact McNemar
  test** p-value (a two-sided binomial test on the discordant pairs), and a paired bootstrap
  95% CI for the difference.

With 164 problems, a 3-point difference is often noise. These functions tell you whether yours is.

In [ ]:
def bootstrap_ci(values, n_boot=10_000, seed=0):
    raise NotImplementedError("TODO 5")


def paired_compare(a, b, n_boot=10_000, seed=0):
    """Return dict: n, a_only, b_only, diff (pass rate A - B), p_mcnemar, ci_low, ci_high."""
    raise NotImplementedError("TODO 5")

### Run everything

This is the long cell. With `QUICK = True` it takes minutes; with `QUICK = False` it takes
about 2.5 hours on a T4 (E0–E2), so start it early. Re-running it after an error resumes from
the cache; a disconnected runtime does not (the local disk is wiped).

In [ ]:
SAMPLES.clear(); RESULTS.clear(); EFFICIENCY.clear(); PASS.clear(); E2_OUT.clear()
t0 = time.time()
run_all()
print(f"\nall configurations done in {(time.time() - t0) / 60:.1f} min")

record_e0()
record_e1()
if RUN_E2:
    record_e2()
if RUN_E3:
    record_e3()

results = pd.DataFrame(RESULTS)
pd.DataFrame(EFFICIENCY)

In [ ]:
# E0 and E1 at a glance
view = results[results["metric"].isin(["pass@1", "robust_pass", "robust_drop", "conditional_robust"])]
view.pivot_table(index=["eval", "variant", "metric"], columns="config", values="value")[CONFIGS].round(3)

### Paired comparisons against `fp16` (provided once TODO 5 works)

Each quantized configuration is compared with `fp16` on every nominal set and every
perturbation. Read the discordant counts, not only the p-value.

In [ ]:
rows = []
for config in CONFIGS[1:]:
    for name in list(NOMINAL_FILES) + [f"{f}/{v}" for f, vs in RECODE_SUBSET.items() for v in vs]:
        c = paired_compare(PASS["fp16"][name], PASS[config][name])
        rows.append({"vs": config, "dataset": name, **c})
        record_result(config, "paired_vs_fp16", name, "diff_pass@1", c["n"], c["diff"], c["ci_low"], c["ci_high"])
        record_result(config, "paired_vs_fp16", name, "p_mcnemar", c["n"], c["p_mcnemar"])
paired = pd.DataFrame(rows)
paired.round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
e1 = results[(results["eval"] == "E1") & results["variant"].str.endswith("worst_case")]
for ax, metric, title in [(axes[0], "robust_pass", "Robust Pass@1 (family worst case)"),
                          (axes[1], "conditional_robust", "Conditional robustness (family worst case)")]:
    t = e1[e1["metric"] == metric].pivot(index="variant", columns="config", values="value")[CONFIGS]
    t.index = [v.split("/")[0] for v in t.index]
    t.plot.bar(ax=ax, rot=0)
    ax.set_title(title); ax.set_ylim(0, 1); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "e1_by_family.png"), dpi=150)
plt.show()

## 9. Part 6 — Report (15 pts)

Write the report in the markdown cells below. Short and specific beats long. Back every
claim with a number from `results` or `paired`.

**The confound you must handle.** A configuration that solves fewer nominal problems has
fewer problems left to break, so its *absolute* robust pass can drop less — and it can look
"more robust" for the wrong reason. Example: `fp16` solves 100 nominal problems and keeps
80 under a perturbation (20 lost); `nf4` solves 70 and keeps 60 (10 lost). Only 10 lost
looks better, yet conditional robustness is 80% for `fp16` and 86% for `nf4` — and whether
*that* gap is real is a question for Part 5. Report absolute and conditional numbers side by
side, and discuss any case where they disagree.

### 9.1 Quantization

*Library versions; for `gptq4`: calibration dataset, number of samples, sequence length, and
why you chose them. Efficiency table (size, memory, tokens/s) and what it does and does not
show (see the note on GPTQ decompression).*

TODO

### 9.2 Results

*E0 and E1 per configuration, both absolute and conditional. Which perturbation families,
if any, does quantization make worse? Do `nf4` and `gptq4` differ? Which differences
survive the paired test, and which are within noise? Then E2 (ASL and the per-loop curve)
and/or E3 (spread across templates).*

TODO

### 9.3 Three flipped examples

*At least three problems that the `fp16` model solves under a perturbation and a quantized
model does not (or the reverse). For each: the perturbation, both completions (find them in
`SAMPLES`), and your explanation of what went wrong.*

TODO

In [ ]:
# helper: find flips for one perturbation and show both completions
def flips(dataset, a="fp16", b="nf4", limit=5):
    pa, pb = PASS[a][dataset], PASS[b][dataset]
    ids = [t for t in pa if pa[t] and not pb[t]][:limit]
    for t in ids:
        for config in (a, b):
            s = next(s for s in SAMPLES if s["config"] == config and s["variant"] == dataset and s["item_id"] == t)
            print(f"===== {t} [{config}] {s['status']}")
            print(s["prompt"] + s["output"])
    return ids

flips("func_name/CamelCase")

### 9.4 Conclusion and threats to validity

*Your answer to the assignment's question, at the confidence your data supports. What could
make it wrong: one model, one seed per transformation, greedy decoding, the subset of
perturbations, the calibration set, decompressed GPTQ inference, and anything else you noticed.*

TODO

### 9.5 AI assistance

*What you used and for what.*

TODO

## 10. Part 7 — Submit

Run the check below. It only checks that the run is complete, not that it is correct.

Then upload **two files** to Gradescope:

1. **This notebook, with all its outputs.** Run it top to bottom with `QUICK = False`, then
   *File → Download → Download .ipynb*. A notebook without outputs cannot be graded.
2. **`README.md`** explaining your code. Keep it short — a few lines per item:
   - how to run the notebook, and the library versions you used (Part 0 prints them);
   - for each TODO: what your implementation does and any choice you made — in particular
     the GPTQ calibration dataset, number of samples and sequence length, and why;
   - every place you departed from a paper's protocol (ReCode, EVALOOOP, Sclar et al.), and why;
   - which of E2 / E3 you ran;
   - any material AI assistance.

The report itself (results, flipped examples, conclusion) stays in Part 6 of the notebook.

In [ ]:
def check_complete():
    df = pd.DataFrame(RESULTS)
    problems = []
    if QUICK:
        problems.append("QUICK is still True: the graded run must use all 164 problems")
    for config in CONFIG_META:
        if not ((df["config"] == config) & (df["eval"] == "E0")).any():
            problems.append(f"no E0 result for {config}")
        for family in RECODE_SUBSET:
            if not ((df["config"] == config) & (df["variant"] == f"{family}/worst_case")).any():
                problems.append(f"no E1 {family} result for {config}")
    if not (df["eval"].isin(["E2", "E3"])).any():
        problems.append("neither E2 nor E3 was run")
    if df["value"].isna().any():
        problems.append("some result values are missing")
    print("\n".join(problems) or "run complete - download the notebook and write your README")
    return not problems


check_complete()